In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
df = pd.read_csv("tabula-pv_resultats_sp_info1.csv")


: 

In [ ]:
df.head()

In [ ]:
df.info()

## **Cleaning w Merging**

In [ ]:
df.columns = df.columns.str.strip()
df = df.dropna(axis=1, how='all')
df = df.dropna(subset=["Ident."])
df.drop("N°",axis=1,inplace=True)
df[["Ident.", "Prenom", "Nom"]] = df[["Ident.", "Prenom", "Nom"]].fillna(method="ffill")

In [ ]:
df['row_number'] = df.groupby('Ident.').cumcount()
grade_blocks = []
unique_row_counts = df['row_number'].max() + 1
for i in range(unique_row_counts):
    block = df[df['row_number'] == i].copy()
    grade_cols = [col for col in block.columns if col not in ["Ident.", "Prenom", "Nom", 'row_number']]
    block.rename(columns={col: f'Block{i+1}_{col}' for col in grade_cols}, inplace=True)
    block = block.drop(columns=['row_number'])
    grade_blocks.append(block)
from functools import reduce
merged_df = reduce(lambda left, right: pd.merge(left, right, on=["Ident.", "Prenom", "Nom"], how='outer'), grade_blocks)
merged_df = merged_df.iloc[:, :75]

## **Data Manipulation**

In [ ]:
merged_df.columns=['Ident.','Nom','Prenom','cc_math_ing', 'exam_math_ing','moy_math_ing','cc_analyse_s1', 'exam_analyse_s1','moy_analyse_s1','cc_algo', 'exam_algo','moy_algo','cc_prog', 'exam_prog','moy_prog','cc_TIC', 'exam_TIC','moy_TIC','cc_logique', 'exam_logique','moy_logique','cc_GL', 'exam_GL','moy_GL','cc_circuit', 'exam_circuit','moy_circuit','cc_semi', 'exam_semi','moy_semi','cc_eco_s1', 'exam_eco_s1','moy_eco_s1','cc_ang_s1', 'exam_ang_s1','moy_ang_s1','cc_fr_s1', 'exam_fr_s1','moy_fr_s1','cc_proba/stat', 'exam_proba/stat','moy_proba/stat','cc_analyse_s2', 'exam_analyse_s2','moy_analyse_s2','cc_structure', 'exam_structure','moy_structure','cc_POO', 'exam_POO','moy_POO','cc_archi', 'exam_archi','moy_archi','cc_reseau', 'exam_reseau','moy_reseau','cc_web', 'exam_web','moy_web','cc_BD', 'exam_BD','moy_BD','cc_conception', 'exam_conception','moy_conception','cc_eco_s2', 'exam_eco_s2','moy_eco_s2','cc_ang_s2', 'exam_ang_s2','moy_ang_s2','cc_fr_s2', 'exam_fr_s2','moy_fr_s2']
merged_df.sort_values('Nom',inplace=True)
merged_df.reset_index(drop=True,inplace=True)

In [ ]:
for col in merged_df.columns[3:]:
    merged_df[col] = pd.to_numeric(merged_df[col].astype(str).str.replace(',', '.'), errors='coerce')

In [ ]:
coef_math_ing = 3
coef_analyse_s1 = 2
coef_algo = 3
coef_prog = 4
coef_TIC = 2
coef_logique = 3
coef_GL = 3
coef_circuit = 3
coef_semi = 2
coef_eco_s1 = 2
coef_ang_s1 = 1.5
coef_fr_s1 = 1.5
coef_eco_s2 = 2
coef_ang_s2 = 1.5
coef_fr_s2 = 1.5
coef_proba_stat = 3
coef_analyse_s2 = 2
coef_structure = 2.5
coef_POO = 4
coef_archi = 2.5
coef_reseau = 2.5
coef_conception = 3.5
coef_BD = 2.5
coef_web = 2.5

merged_df['moyenne_annuelle'] = ((merged_df['moy_math_ing'] * coef_math_ing +merged_df['moy_analyse_s1'] * coef_analyse_s1 +merged_df['moy_algo'] * coef_algo +merged_df['moy_prog'] * coef_prog +merged_df['moy_TIC'] * coef_TIC +merged_df['moy_logique'] * coef_logique +merged_df['moy_GL'] * coef_GL +merged_df['moy_circuit'] * coef_circuit +merged_df['moy_semi'] * coef_semi +merged_df['moy_eco_s1'] * coef_eco_s1 +merged_df['moy_ang_s1'] * coef_ang_s1 +merged_df['moy_fr_s1'] * coef_fr_s1 +merged_df['moy_proba/stat'] * coef_proba_stat +merged_df['moy_analyse_s2'] * coef_analyse_s2 +merged_df['moy_structure'] * coef_structure +merged_df['moy_POO'] * coef_POO +merged_df['moy_archi'] * coef_archi +merged_df['moy_reseau'] * coef_reseau +merged_df['moy_eco_s2'] * coef_eco_s2 +merged_df['moy_fr_s2'] * coef_fr_s2 +merged_df['moy_ang_s2'] * coef_ang_s2 +merged_df['moy_conception'] * coef_conception +merged_df['moy_web'] * coef_web  +merged_df['moy_BD'] * coef_BD )/(coef_math_ing + coef_analyse_s1 + coef_algo + coef_prog + coef_TIC + coef_logique + coef_GL + coef_circuit + coef_semi + coef_eco_s1 + coef_ang_s1 + coef_fr_s1 + coef_proba_stat + coef_analyse_s2 + coef_structure + coef_POO + coef_archi + coef_reseau + coef_BD + coef_conception + coef_ang_s2 +  coef_eco_s2 + coef_fr_s2 + coef_web)).round(2)

merged_df

In [ ]:
Classe_A=pd.read_csv('Classe_A.csv')
Classe_A["Classe"]="A"
Classe_A.drop(columns="N°",inplace=True)
Classe_B=pd.read_csv('Classe_B.csv')
Classe_B["Classe"]="B"
Classe_B.drop(columns="N°",inplace=True)
Classe_C=pd.read_csv('Classe_C.csv')
Classe_C["Classe"]="C"
Classe_C.drop(columns="N°",inplace=True)
Classe_D=pd.read_csv('Classe_D.csv')
Classe_D["Classe"]="D"
Classe_D.drop(columns="N°",inplace=True)
Classes=pd.concat([Classe_A,Classe_B,Classe_C,Classe_D])
Classes.reset_index(drop=True,inplace=True)
Classes.columns=['Nom','Prenom','Classe']
Classes['Nom']=Classes['Nom'].str.upper()
Classes['Prenom']=Classes['Prenom'].str.upper()

In [ ]:
merged_df.loc[77,'Nom']="LTAYEF"
merged_df.loc[19,'Prenom']="MOHAMED CHAKER"
merged_df.loc[61,'Prenom']="SAAD EDDINE"
merged_df.loc[67,'Prenom']="BARAA"
merged_df.loc[82,'Prenom']="MOHAMED HOSNI"
merged_df.loc[104,'Prenom']="HOUSSEM EDDINE"
merged_df.loc[8,['Nom','Prenom']]=["ATIG","ABDERAHMEN"]
merged_df.loc[27,'Nom']="BENOTHMEN"
merged_df.loc[47,'Prenom']="MOUHIB"
merged_df.loc[48,'Prenom']="ISLEM"
merged_df.loc[55,'Prenom']="ABDERRAHMEN"
merged_df.loc[76,'Prenom']="AHMED YASSINE"
merged_df.loc[96,'Prenom']="MOHAMED AZIZ"
merged_df.loc[102,'Prenom']="MOHAMED ALI"
merged_df.loc[14,['Nom','Prenom']]=["ABDELHAKIM","BARBARIA"]
merged_df.loc[4,'Prenom']="MOHAMED DHIA ALISLEM"
merged_df.loc[70,'Prenom']="MOHAMED AMINE"
merged_df.loc[88,'Nom']="OULEDOUHIBA"
merged_df.loc[89,'Prenom']="MOHAMED AMINE"
merged_df.loc[93,'Prenom']="BACEM"
merged_df.loc[17,'Prenom']="MOHAMMED BAYREM"
merged_df.loc[24,'Prenom']="MOHAMED ESSEDIK"
merged_df.loc[26,'Nom']="BEN TAHER"
merged_df.loc[63,'Prenom']="MOHAMED AMIN"
merged_df.loc[78,'Prenom']="MOHAMED MOUNIB"
merged_df.loc[84,'Prenom']="MOHAMED ALI"
merged_df.loc[91,'Prenom']="MOHAMED AZIZ"
merged_df.loc[95,'Prenom']="MAKREM"
info_1ere=merged_df.merge(Classes,on=['Nom','Prenom'])
info_1ere

In [ ]:
columns = list(info_1ere.columns)
last_column = columns.pop(-1)
columns.insert(3, last_column)
info_1ere = info_1ere[columns]
info_1ere.to_csv("Info_1ere.csv")
info_1ere.groupby('Classe')['moyenne_annuelle'].mean().plot(kind='bar',xlabel='Classe', ylabel='Moyenne annuelle', title='Moyenne annuelle par classe')
plt.show()
for i in info_1ere.columns:
    if i.startswith('moy_'):
        info_1ere.groupby('Classe')[i].mean().plot(kind='bar')
        plt.xlabel('Classe')
        plt.ylabel('Moyenne de {}'.format(i))
        plt.title('Moyenne de {} par classe'.format(i))
        plt.show()

In [ ]:
admis=pd.read_csv("liste-des-admis.csv")
admis.columns=["Nom","Prenom"]
admis["reussite"]="A"
info_1ere=info_1ere.merge(admis,how='left',on=['Nom','Prenom'])

In [ ]:
print(info_1ere["reussite"].value_counts(),admis["reussite"].value_counts())

In [ ]:
columns = list(info_1ere.columns)
last_column = columns.pop(-1)
columns.insert(4, last_column)
info_1ere=info_1ere[columns]

In [ ]:
lost_students={"AHMED":"LTAYEF","AYA":"CHOKRI","MOUHIB":"FAHEM","NOUR":"BEN TAHER","MOHAMED ESSEDIK":"BEN SASSI"}
info_1ere["reussite"] = np.where(
((info_1ere["Nom"].isin(lost_students.values()) & info_1ere["Prenom"].isin(lost_students.keys())) | (info_1ere["reussite"] == "A")),"A","R")
info_1ere["Niveau"]="1ère année"

# **----------------------------------The final Dataframe----------------------------------**

In [ ]:
info_1ere

# **------------------------------------------------------------------------------------------------**

In [ ]:
info_1ere.to_csv("Info_1ere.csv")

In [ ]:
print(info_1ere["reussite"].value_counts(),admis["reussite"].value_counts())

In [ ]:
info_1ere.columns

In [ ]:
X=info_1ere[[feature for feature in info_1ere.columns[6:41] if feature.startswith('moy_')]]
X_dict={index:item for index,item in enumerate(X) }
X_dict

In [ ]:
y=info_1ere.reussite
y

In [ ]:
X_train, X_test, y_train, y_test=train_test_split(X,y,random_state=0,stratify=y,test_size=0.3)
model=LogisticRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:", classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["No success", "Success"], yticklabels=["No Success", "Success"])
plt.xlabel("Prediction")
plt.ylabel("Reality")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
hazem=info_1ere[info_1ere["Prenom"]=='HAZEM']
hazem_pred=model.predict(hazem[[feature for feature in hazem.columns[6:41] if feature.startswith('moy_')]])
hazem_pred[0]